# Tonic vs phasic EDA synchrony in interacting groups

**Question.** Do the tonic and phasic components of electrodermal activity synchronize to *different* social features during four-person interaction?

**Data.** GroupAffect-4 (public, Zenodo 20809799): 10 groups × 4 participants wearing EmotiBit sensors (~25 Hz EDA/HR/temperature) through five tasks — T0 resting baseline, T1 hidden-profile decision, T2 mini-negotiation, T3 idea generation, T4 public-goods game.

**Design.** (1) Tonic/phasic split *within* each person, then synchrony *between* people — the tonic/phasic distinction carried across the interpersonal boundary. (2) T0 rest as a null/comparison condition. (3) Every synchrony estimate tested against 200 phase-randomized surrogates. (4) A detrended-tonic control for shared drift. (5) Early-task synchrony → post-task engagement (prediction).

Run time: ~6 minutes end to end.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as spsig

sys.path.insert(0, '../src')
from load import load_group_task, list_groups_tasks, TASKS, CHANNELS
from eda import qc_eda, qc_ok, decompose_eda
from synchrony import windowed_pearson, dyad_synchrony, group_synchrony
from surrogates import phase_randomized, empirical_p

RAW = Path('../data/raw')
HZ, WIN_S, STEP_S, N_NULL = 25.0, 60, 10, 200
%matplotlib inline

## 1. Data inventory & channel verification

The EmotiBit LSL files are long-format (`lsl_time, stream_name, stream_type, value_0…value_25`). Channel indices follow the dataset authors' pipeline defaults, verified empirically below: `value_3` behaves like EDA (µS units, slow drift + responses), `value_0` like raw PPG counts, `value_6` like heart rate (bpm), `value_10` like skin temperature.

In [ ]:
pairs = list_groups_tasks(RAW)
print(f'{len(pairs)} group x task sessions')
print('groups:', sorted({g for g, _ in pairs}))

# spot-check channel behavior on one file
parts = load_group_task(RAW, 'grp-07', 'T1')
d = parts['P1']
for c in ['eda', 'ppg', 'hr', 'temp']:
    v = d[c].to_numpy()
    print(f'{c:5s}  mean={np.mean(v):9.2f}  sd={np.std(v):8.2f}  range=[{np.min(v):.2f}, {np.max(v):.2f}]')

## 2. QC

The EmotiBit repeats values across LSL rows (~60% exact-zero diffs at the 25 Hz read rate) — short repeats are normal quantization, **not** dropout. Real dropout signatures: NaN gaps, pinning at exactly 0.030 µS (electrode off), or constant runs ≥ 10 s. Participants failing any of these (>30% missing/railed, >50% stuck) are excluded per session.

In [ ]:
qc_rows = []
for group, task in pairs:
    try:
        parts = load_group_task(RAW, group, task)
    except ValueError:
        continue
    for p, d in parts.items():
        q = qc_eda(d['eda'].to_numpy(), HZ)
        qc_rows.append({'group': group, 'task': task, 'p': p,
                         'ok': qc_ok(d['eda'].to_numpy(), HZ), **q})
qc = pd.DataFrame(qc_rows)
print('participants:', len(qc), '| pass:', qc['ok'].sum(),
      f"({qc['ok'].mean():.0%})  | sessions with >=2 good:",
      qc.groupby(['group', 'task'])['ok'].sum().ge(2).sum(), '/', len(pairs))

## 3. Tonic/phasic decomposition (example)

Zero-phase 4th-order Butterworth split at 0.05 Hz (second-order sections — numerically stable at this low cutoff): tonic = slow drift (shared context / arousal level), phasic = fast event responses.

In [ ]:
eda = parts['P1']['eda'].to_numpy()
comp = decompose_eda(eda, HZ)
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
t = np.arange(len(eda)) / HZ
ax[0].plot(t, eda, lw=0.6); ax[0].set_ylabel('EDA (µS)')
ax[0].set_title('Example participant (grp-07, T1): raw EDA and its decomposition')
ax[1].plot(t, comp['eda_tonic'], lw=0.8, color='C1'); ax[1].set_ylabel('tonic (µS)')
ax[2].plot(t, comp['eda_phasic'], lw=0.6, color='C2'); ax[2].set_ylabel('phasic (µS)')
ax[2].set_xlabel('time (s)')
fig.tight_layout(); fig.savefig('../figures/fig1_example_decomposition.png', dpi=150)
plt.show()

## 4. Synchrony + surrogate testing (all sessions)

Per session: align participants on the common LSL grid → decompose → windowed Pearson r (60 s windows, 10 s steps) per dyad → mean over the 6 dyads = group synchrony curve. Each mean is tested against 200 phase-randomized nulls (identical spectrum/autocorrelation, coupling destroyed). A linearly-detrended tonic control separates genuine co-fluctuation from shared drift.

In [ ]:
win, step = int(WIN_S * HZ), int(STEP_S * HZ)
rows, example = [], {}
for gi, (group, task) in enumerate(pairs):
    try:
        parts = load_group_task(RAW, group, task)
    except ValueError:
        continue
    good = {p: d for p, d in parts.items() if qc_ok(d['eda'].to_numpy(), HZ)}
    if len(good) < 2:
        print(f'skip {group} {task}: only {len(good)} good participants'); continue
    comp = {p: decompose_eda(d['eda'].to_numpy(), HZ) for p, d in good.items()}
    tonic = {p: c['eda_tonic'].to_numpy() for p, c in comp.items()}
    phasic = {p: c['eda_phasic'].to_numpy() for p, c in comp.items()}
    tonic_dt = {p: spsig.detrend(t) for p, t in tonic.items()}
    g_t = group_synchrony(dyad_synchrony(tonic, win, step))
    g_p = group_synchrony(dyad_synchrony(phasic, win, step))
    g_d = group_synchrony(dyad_synchrony(tonic_dt, win, step))
    m_t, m_p, m_d = (float(np.nanmean(g)) for g in (g_t, g_p, g_d))
    rng = np.random.default_rng(hash((group, task)) % 2**32)
    null_t, null_p, null_d = np.empty(N_NULL), np.empty(N_NULL), np.empty(N_NULL)
    keys = sorted(tonic)
    for i in range(N_NULL):
        null_t[i] = np.nanmean(group_synchrony(dyad_synchrony({k: phase_randomized(tonic[k], rng) for k in keys}, win, step)))
        null_p[i] = np.nanmean(group_synchrony(dyad_synchrony({k: phase_randomized(phasic[k], rng) for k in keys}, win, step)))
        null_d[i] = np.nanmean(group_synchrony(dyad_synchrony({k: phase_randomized(tonic_dt[k], rng) for k in keys}, win, step)))
    p_t, p_p, p_d = empirical_p(m_t, null_t), empirical_p(m_p, null_p), empirical_p(m_d, null_d)
    rows.append(dict(group=group, task=task, n=len(comp), m_tonic=m_t, p_tonic=p_t,
                       m_phasic=m_p, p_phasic=p_p, m_tonic_dt=m_d, p_tonic_dt=p_d))
    if group == 'grp-07' and task == 'T1':
        example = dict(g_t=g_t, g_p=g_p, null_t=null_t, m_t=m_t)
    print(f'[{gi+1}/{len(pairs)}] {group} {task} (n={len(comp)}): '
          f'tonic r={m_t:.3f} p={p_t:.3f} | phasic r={m_p:.3f} p={p_p:.3f} | tonic_detr r={m_d:.3f} p={p_d:.3f}')
res = pd.DataFrame(rows)
res.to_csv('../results/synchrony_summary.csv', index=False)
print('saved', len(res), 'sessions')

## 5. Task contrasts

T0 (rest) is the comparison condition: any synchrony there reflects shared environment, not interaction.

In [ ]:
r = res[np.isfinite(res['m_tonic'])].copy()
g = r.groupby('task').agg(mt=('m_tonic', 'mean'), md=('m_tonic_dt', 'mean'), mp=('m_phasic', 'mean'),
                            st=('m_tonic', 'sem'), sd=('m_tonic_dt', 'sem'), sp=('m_phasic', 'sem'),
                            sig_t=('p_tonic', lambda s: (s < 0.05).sum()), n=('group', 'count')).reindex(TASKS)
print(g.round(3).to_string())
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(TASKS)); w = 0.25
ax.bar(x - w, g['mt'], w, yerr=g['st'].fillna(0), label='tonic', capsize=3)
ax.bar(x, g['md'], w, yerr=g['sd'].fillna(0), label='tonic (detrended)', capsize=3)
ax.bar(x + w, g['mp'], w, yerr=g['sp'].fillna(0), label='phasic', capsize=3)
ax.set_xticks(x); ax.set_xticklabels(TASKS)
ax.set_ylabel('mean group synchrony (r ± SEM)')
ax.set_title('Tonic vs phasic synchrony by task (T0 = resting baseline)')
ax.legend(); fig.tight_layout()
fig.savefig('../figures/fig3_task_contrast.png', dpi=150); plt.show()

## 6. Is the synchrony real? (surrogate example)

One session's observed group synchrony against its 200 phase-randomized nulls:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
tt = np.arange(len(example['g_t'])) * STEP_S / 60
ax.plot(tt, example['g_t'], label='tonic synchrony', lw=1.5)
ax.plot(tt, example['g_p'], label='phasic synchrony', lw=1.5)
ax.set_xlabel('time (min)'); ax.set_ylabel('group synchrony (r)')
ax.set_title('Group synchrony over time, grp-07 T1 (hidden-profile decision)')
ax.legend(); fig.tight_layout()
fig.savefig('../figures/fig2_synchrony_curves.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(example['null_t'], bins=30, alpha=0.7, label='phase-randomized null')
ax.axvline(example['m_t'], color='red', lw=2, label=f"observed r={example['m_t']:.3f}")
ax.set_xlabel('group tonic synchrony'); ax.set_ylabel('count')
ax.set_title('Surrogate test: is the synchrony real?'); ax.legend()
fig.tight_layout(); fig.savefig('../figures/fig4_surrogate_null.png', dpi=150); plt.show()

## 7. Prediction: early synchrony → post-task engagement

Does synchrony in the *first 3 minutes* predict the group's post-task engagement rating? Small-n (≤10 groups) — reported as exploratory with exact p-values.

In [ ]:
import glob
from scipy import stats as sps
dfs = [pd.read_csv(f, sep='\t') for f in sorted(glob.glob(str(RAW / 'beh/bids_release_no_video/*/ses-*_grp-*/beh/*stimuli_answers.tsv')))]
beh = pd.concat(dfs, ignore_index=True)
beh['grp'] = beh['source_file'].str.extract(r'(grp-\d+)')
eng = beh[beh['item_key'] == 'engagement'].copy()
eng['val'] = pd.to_numeric(eng['item_value'], errors='coerce')
eng_g = eng.groupby(['grp', 'task'])['val'].mean().reset_index()

prow = []
for _, er in eng_g.iterrows():
    try:
        parts = load_group_task(RAW, er['grp'], er['task'])
    except ValueError:
        continue
    good = {p: d for p, d in parts.items() if qc_ok(d['eda'].to_numpy(), HZ)}
    if len(good) < 2:
        continue
    n3 = int(180 * HZ)
    tc, pc = {}, {}
    for p, d in good.items():
        e = d['eda'].to_numpy()[:n3]
        if len(e) < n3 * 0.8:
            break
        c = decompose_eda(e, HZ)
        tc[p], pc[p] = c['eda_tonic'].to_numpy(), c['eda_phasic'].to_numpy()
    else:
        w3, s3 = int(60 * HZ), int(10 * HZ)
        prow.append({'group': er['grp'], 'task': er['task'],
                       'early_tonic': float(np.nanmean(group_synchrony(dyad_synchrony(tc, w3, s3)))),
                       'early_phasic': float(np.nanmean(group_synchrony(dyad_synchrony(pc, w3, s3)))),
                       'engagement': er['val']})
        continue
pred = pd.DataFrame(prow)
pred.to_csv('../results/prediction.csv', index=False)
for comp, lab in [('early_tonic', 'tonic'), ('early_phasic', 'phasic')]:
    m = np.isfinite(pred[comp]) & np.isfinite(pred['engagement'])
    rr, pp = sps.pearsonr(pred.loc[m, comp], pred.loc[m, 'engagement'])
    print(f'early {lab} synchrony vs engagement: r={rr:.3f}, p={pp:.3f} (n={m.sum()})')
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for t, dd in pred[m].groupby('task'):
        ax.scatter(dd[comp], dd['engagement'], label=t)
    ax.set_xlabel(f'early {lab} synchrony (first 3 min)')
    ax.set_ylabel('post-task group engagement')
    ax.set_title(f'Early {lab} synchrony → engagement'); ax.legend()
    fig.tight_layout(); fig.savefig(f'../figures/fig5_prediction_{lab}.png', dpi=150); plt.show()

## 8. Directed coupling: who drives whom?

Undirected correlation can't say who influences whom. Here: Granger causality on tonic EDA (1 Hz) in *both* directions per dyad, each F-statistic tested against 200 time-shifted nulls (circular shifts destroy coupling, preserve everything else — the right null for directionality). Lead/lag timescale comes from cross-correlating *differenced* tonic: on drift-dominated signals the raw xcorr peaks at lag 0 no matter what. Run with `python ../scripts/run_directed.py`; results below are the saved outputs.


In [ ]:
d = pd.read_csv('../results/directed_coupling.csv')
d['a_drives'] = (d['p_a_to_b'] < 0.05) & ~(d['p_b_to_a'] < 0.05)
d['b_drives'] = (d['p_b_to_a'] < 0.05) & ~(d['p_a_to_b'] < 0.05)
d['directed'] = d['a_drives'] | d['b_drives']
print(f"directed dyads: {d['directed'].sum()}/{len(d)} ({d['directed'].mean():.1%}; ~9.5% expected under null)")
print(d.groupby('task')['directed'].mean().reindex(TASKS).round(3).to_string())
dd = d[d['directed']].copy()
dd['driver_lead_s'] = np.where(dd['a_drives'], dd['lead_x_s'], -dd['lead_x_s'])
print(f"driver lead: median {dd['driver_lead_s'].median():.0f}s, IQR [{dd['driver_lead_s'].quantile(.25):.0f}, {dd['driver_lead_s'].quantile(.75):.0f}]")


Directed coupling exists above chance (~21% vs ~10% null) but is flat across tasks and has no stable lead timescale. The social hypothesis — the most dominant member drives — fails cleanly:


In [ ]:
lnk = pd.read_csv('../results/dominance_driver.csv')
m, n = lnk['match'].sum(), len(lnk)
from scipy.stats import binomtest
print(f"most dominant == top driver: {m}/{n} (chance 25%), binomial p={binomtest(m, n, 0.25, alternative='greater').pvalue:.3f}")
print(f"median dominance-driver rank correlation: {lnk['spearman'].median():.3f}")


## 9. Findings

**1. Tonic synchrony is a resting phenomenon, not an interaction phenomenon.** Mean group tonic synchrony at rest (T0): r = 0.11, significant vs phase-randomized nulls in 3 of 7 groups (grp-07: r = 0.33, p = .005; grp-13: r = 0.25, p = .015; grp-14: r = 0.26, p = .005). During the four interaction tasks the mean collapses to ≈ 0 (T1: 0.03, T2: 0.01, T3: −0.03, T4: 0.03), with only 2 of 32 sessions significant — chance level. At rest all four participants share the same context (same room, same posture, same nothing-to-do), so slow arousal drifts together; differentiated task roles break the coupling. The resting 'baseline' is not a null — it is the condition of *maximal* shared context, which flips the usual hyperscanning logic (task > rest = social coupling).

**2. Much of the resting coupling is shared drift.** After linear detrending, mean T0 tonic synchrony drops from 0.11 to 0.05 (3/7 → 2/7 significant). Phase-randomized nulls preserve autocorrelation but not co-trending — the detrended control is the honest estimate, and it says: part drift, part genuine co-fluctuation.

**3. Phasic synchrony is ≈ 0 everywhere.** Mean phasic r per task sits between −0.01 and −0.006; no task exceeds chance. Fast sympathetic responses do not align across group members at 60 s resolution — a boundary condition for 'shared arousal' claims.

**4. Early phasic synchrony predicts engagement.** First-3-minutes phasic synchrony vs post-task group engagement (20 group × task observations): Pearson r = 0.61, p = .004; Spearman r = 0.49, p = .030; leave-one-out r ∈ [0.47, 0.66] (no single point drives it). Group-averaged over 7 independent groups: r = 0.67, p = .098 — same direction, underpowered. So while *average* phasic synchrony is nil, groups whose fast responses align in the opening minutes end up more engaged: a temporal, predictive effect.

Together: tonic and phasic EDA synchronize to different social features after all — tonic tracks shared context (maximal at rest), phasic tracks the interaction's opening moments and forecasts where the group ends up.

## 9. Limitations & reproducibility

- Channel indices (`value_3` = EDA etc.) follow the dataset authors' pipeline defaults and were confirmed empirically; not manufacturer-certified.
- Sessions lost to sensor dropout are reported in the QC table, not silently dropped.
- Phase-randomized nulls control for autocorrelation but not shared linear drift — hence the detrended-tonic control.
- The prediction analysis is exploratory at n ≤ 10 groups.
- Seeds: surrogate RNGs are seeded per (group, task) via `hash((group, task))`; reruns are bit-identical.
- `pip install -r requirements.txt` + `bash data/download_groupaffect4.sh` + Run All reproduces every number and figure.
**5. Directed coupling is real but weak, context-free, and unrelated to dominance.** ~21% of dyads show directed tonic coupling (Granger, shift-nulls) vs ~10% under the null, flat across tasks (16–27%); median driver lead 6 s with IQR spanning zero — no stable leader-follower timescale. The most dominant member is the top driver in 4/27 group-tasks (chance 25%, p = .93): an honest null.
